In [1]:
%pip install --force-reinstall "numpy==1.26.4" "pandas==2.2.3" geopandas scikit-learn

  Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.3-cp312-cp312-macosx_10_9_x86_64.whl.metadata (89 kB)
  Using cached geopandas-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-macosx_10_13_x86_64.whl.metadata (9.3 kB)
  Using cached pytz-2026.6-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.6-py2.py3-none-any.whl.metadata (1.4 kB)
INFO: pip is looking at multiple versions of geopandas to determine which version is compatible with other requirements. This could take a while.
  Using cached geopandas-1.1.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-macosx_12_0_x86_64.whl.metadata (5.8 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyproj-3.8.0-cp312-cp312-macosx_15_0_x86_64.whl.metadata (40 kB)
  Using cached shapely-2.2.0-cp312-cp312-macosx_10_13_x86_64.whl.metadata (6.7 kB)
  Using cached scipy-1.18.1-cp3

In [10]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [25]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

print("Current folder:", Path.cwd())

# Find all CRMLSSold CSV files in the project folder and subfolders.
files = sorted(
    Path.cwd().rglob("CRMLSSold*.csv")
)

# Exclude the separate 2022–2023 file.
files = [
    file for file in files
    if "20220101_20231231" not in file.name
]

if not files:
    raise FileNotFoundError(
        "No CRMLSSold CSV files were found in the project folder."
    )

print("Files found:")
for file in files:
    print(file)

# Read each monthly file.
frames = []

for file in files:
    temp = pd.read_csv(file, low_memory=False)
    temp["source_file"] = file.name
    frames.append(temp)

# Combine all monthly files.
df = pd.concat(
    frames,
    ignore_index=True,
    sort=False
)

# Convert CloseDate into a date.
df["CloseDate"] = pd.to_datetime(
    df["CloseDate"],
    errors="coerce"
)

# Keep only residential single-family properties.
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

print("Rows after property filtering:", len(df))

Current folder: /Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks
Files found:
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202406_filled.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202407_filled.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202409.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202410.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202411.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202412.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202501_filled.csv
Rows after property filtering: 78639


In [19]:
# These are the original numeric columns.
original_numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "DaysOnMarket",
    "ListPrice",
    "Latitude",
    "Longitude"
]

target = "ClosePrice"

# Convert numeric columns to numbers.
# Invalid values are changed to missing values.
for column in original_numeric_features + [target]:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# Remove records without a valid positive closing price.
df = df[
    df[target].notna() &
    (df[target] > 0)
].copy()

# Remove duplicate records with a valid ListingKey.
# Keep rows without a ListingKey.
df = df.sort_values(["CloseDate", "ListingKey"])

with_key = (
    df[df["ListingKey"].notna()]
    .drop_duplicates("ListingKey", keep="last")
)

without_key = df[df["ListingKey"].isna()]

df = pd.concat(
    [with_key, without_key],
    ignore_index=True
)

# Create a month label for the time-based split.
df["CloseMonth"] = (
    df["CloseDate"]
    .dt.to_period("M")
    .astype(str)
)

print("Prepared rows:", len(df))

Prepared rows: 78580


In [20]:
# Create a bed-to-bathroom ratio.
# Replace zero bathrooms with missing values to avoid division by zero.
df["BedBathRatio"] = (
    df["BedroomsTotal"] /
    df["BathroomsTotalInteger"].replace(0, np.nan)
)

# Calculate the property's age when it was sold.
df["CloseYear"] = df["CloseDate"].dt.year
df["PropertyAge"] = df["CloseYear"] - df["YearBuilt"]

# Negative ages are invalid, so replace them with missing values.
df.loc[df["PropertyAge"] < 0, "PropertyAge"] = np.nan

print(df[[
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "BedBathRatio",
    "YearBuilt",
    "PropertyAge"
]].head())

   BedroomsTotal  BathroomsTotalInteger  BedBathRatio  YearBuilt  PropertyAge
0            2.0                    2.0      1.000000     1985.0         39.0
1            4.0                    3.0      1.333333     2003.0         21.0
2            5.0                    5.0      1.000000     2016.0          8.0
3            5.0                    3.0      1.666667     1992.0         32.0
4            5.0                    3.0      1.666667     1990.0         34.0


In [26]:
# Search for the extracted school-district shapefile.
district_files = sorted(Path.cwd().rglob("*.shp"))

if not district_files:
    raise FileNotFoundError(
        "No school-district shapefile was found. "
        "Download and unzip it into the project folder."
    )

district_file = district_files[0]

print("School-district file:")
print(district_file)

# Read the district boundaries.
districts = gpd.read_file(district_file)

print("School-district columns:")
print(districts.columns.tolist())

School-district file:
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/DistrictAreas2425/DistrictAreas2425.shp
School-district columns:
['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'GradeLowCe', 'GradeHighC', 'AssistStat', 'CongressUS', 'SenateCA', 'AssemblyCA', 'UpdateNote', 'EnrollTota', 'EnrollChar', 'EnrollNonC', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAre', 'Shape__Are', 'Shape__Len', 'geometry']


In [27]:
print("Shapefiles found:")
for file in district_files:
    print(file)

print("\nColumns in the selected shapefile:")
print(districts.columns.tolist())

print("\nFirst five rows:")
display(districts.head())

Shapefiles found:
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/DistrictAreas2425/DistrictAreas2425.shp

Columns in the selected shapefile:
['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'GradeLowCe', 'GradeHighC', 'AssistStat', 'CongressUS', 'SenateCA', 'AssemblyCA', 'UpdateNote', 'EnrollTota', 'EnrollChar', 'EnrollNonC', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAre', 'Shape__Are', 'Shape__Len', 'geometry']

First five rows:


,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictNa,DistrictTy,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAre,Shape__Are,Shape__Len,geometry
0,1,2024-25,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1356,12.6,3958,36.7,11.248939,4.755489e+07,56522.982683,"MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,2,2024-25,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,357,9.7,1184,32.1,1.789984,7.096327e+06,12696.382797,"POLYGON ((-13612893.866 4565099.707, -13612896..."
2,3,2024-25,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1111,12.1,2686,29.3,10.434329,4.364648e+07,43695.341538,"POLYGON ((-13609482.48 4565074.597, -13609483...."
3,4,2024-25,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,0,0.0,1113,11.6,3728,39.0,66.885571,2.838285e+08,142492.767565,"MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,5,2024-25,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,81,13.7,412,69.8,1.273929,5.363392e+06,13741.272894,"POLYGON ((-13613999.038 4555592.769, -13614126..."


In [28]:
# The shapefile uses shortened column names:
# DistrictNa = school-district name
# DistrictTy = school-district type

district_name_column = "DistrictNa"
district_type_column = "DistrictTy"

print("District name column:", district_name_column)
print("District type column:", district_type_column)

# Create a unique ID for each property.
df["property_id"] = np.arange(len(df))

# Convert each property's longitude and latitude into a geographic point.
property_points = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(
        df["Longitude"],
        df["Latitude"]
    ),
    crs="EPSG:4326"
)

# Make sure the school-district boundaries use the same coordinate system.
districts = districts.to_crs("EPSG:4326")

# Keep only the school-district information needed.
districts_small = districts[
    [
        district_name_column,
        district_type_column,
        "geometry"
    ]
].copy()

# Match each property point to the school-district boundary
# that contains the property.
joined = gpd.sjoin(
    property_points,
    districts_small,
    how="left",
    predicate="within"
)

# Create one readable school-district label.
joined["SchoolDistrict"] = (
    joined[district_name_column].astype("string")
    + " ("
    + joined[district_type_column].astype("string")
    + ")"
)

# Some properties may match multiple district types.
# Combine all matches into one value per property.
district_labels = (
    joined.groupby("property_id")["SchoolDistrict"]
    .agg(lambda values: "|".join(
        sorted(set(value for value in values.dropna()))
    ))
    .rename("SchoolDistrict")
    .reset_index()
)

# Remove the feature if the cell is being re-run.
df = df.drop(columns=["SchoolDistrict"], errors="ignore")

# Add the school-district feature back to the real-estate data.
df = df.merge(
    district_labels,
    on="property_id",
    how="left"
)

# Remove the temporary ID column.
df = df.drop(columns=["property_id"])

print("Properties with school-district information:")
print(df["SchoolDistrict"].notna().sum())

print("Properties missing school-district information:")
print(df["SchoolDistrict"].isna().sum())

print(df[["SchoolDistrict"]].head())

District name column: DistrictNa
District type column: DistrictTy
Properties with school-district information:
78639
Properties missing school-district information:
0
                                      SchoolDistrict
0  Menlo Park City Elementary (Elementary)|Sequoi...
1         Hermosa Beach City Elementary (Elementary)
2  Salinas City Elementary (Elementary)|Salinas U...
3                      Los Angeles Unified (Unified)
4                          Bonsall Unified (Unified)


In [29]:
# Create a bed-to-bathroom ratio.
df["BedBathRatio"] = (
    df["BedroomsTotal"] /
    df["BathroomsTotalInteger"].replace(0, np.nan)
)

# Calculate the property's age when it was sold.
df["CloseYear"] = df["CloseDate"].dt.year
df["PropertyAge"] = df["CloseYear"] - df["YearBuilt"]

# Replace impossible negative ages with missing values.
df.loc[df["PropertyAge"] < 0, "PropertyAge"] = np.nan

print(df[
    [
        "BedBathRatio",
        "PropertyAge",
        "SchoolDistrict"
    ]
].head())

   BedBathRatio  PropertyAge  \
0           1.5         76.0   
1           1.0          0.0   
2           2.0         74.0   
3           3.0        101.0   
4           1.0         15.0   

                                      SchoolDistrict  
0  Menlo Park City Elementary (Elementary)|Sequoi...  
1         Hermosa Beach City Elementary (Elementary)  
2  Salinas City Elementary (Elementary)|Salinas U...  
3                      Los Angeles Unified (Unified)  
4                          Bonsall Unified (Unified)  


In [30]:
# Original feature set from Week 5.
original_numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "DaysOnMarket",
    "ListPrice",
    "Latitude",
    "Longitude"
]

original_categorical_features = [
    "City",
    "CountyOrParish"
]

# New feature set for Week 6.
new_numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "DaysOnMarket",
    "ListPrice",
    "Latitude",
    "Longitude",
    "BedBathRatio",
    "PropertyAge"
]

new_categorical_features = [
    "City",
    "CountyOrParish",
    "SchoolDistrict"
]

# Table comparing the old and new features.
feature_comparison = pd.DataFrame({
    "Feature Set": [
        "Original Week 5 features",
        "New Week 6 features"
    ],
    "Features Added": [
        "None",
        "BedBathRatio, PropertyAge, SchoolDistrict"
    ],
    "Number of Numeric Features": [
        len(original_numeric_features),
        len(new_numeric_features)
    ],
    "Number of Categorical Features": [
        len(original_categorical_features),
        len(new_categorical_features)
    ]
})

feature_comparison

,Feature Set,Features Added,Number of Numeric Features,Number of Categorical Features
0,Original Week 5 features,None,9,2
1,New Week 6 features,"BedBathRatio, PropertyAge, SchoolDistrict",11,3


In [31]:
def create_preprocessor(
    numeric_columns,
    categorical_columns
):
    """
    Creates preprocessing steps for numeric
    and categorical columns.
    """

    return ColumnTransformer([
        (
            "numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                ),
                (
                    "scaler",
                    StandardScaler()
                )
            ]),
            numeric_columns
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="Unknown"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            categorical_columns
        )
    ])

In [35]:
df["CloseDate"] = pd.to_datetime(
    df["CloseDate"],
    errors="coerce"
)

df["CloseMonth"] = (
    df["CloseDate"]
    .dt.to_period("M")
    .astype(str)
)

print(df[["CloseDate", "CloseMonth"]].head())

   CloseDate CloseMonth
0 2024-06-21    2024-06
1 2024-06-25    2024-06
2 2024-06-28    2024-06
3 2024-06-26    2024-06
4 2024-06-28    2024-06


In [36]:
target = "ClosePrice"
test_month = "2025-01"

available_training_months = sorted(
    month
    for month in df["CloseMonth"].dropna().unique()
    if month < test_month
)

train_months = available_training_months[-6:]

train = df[
    df["CloseMonth"].isin(train_months)
].copy()

test = df[
    df["CloseMonth"] == test_month
].copy()

print("Training months:", train_months)
print("Training rows:", len(train))
print("Test rows:", len(test))

Training months: ['2024-06', '2024-07', '2024-09', '2024-10', '2024-11', '2024-12']
Training rows: 70495
Test rows: 8144


In [39]:
target = "ClosePrice"

# Convert ClosePrice to numeric.
df[target] = pd.to_numeric(
    df[target],
    errors="coerce"
)

# Keep only rows with a valid positive closing price.
df = df[
    df[target].notna() &
    (df[target] > 0)
].copy()

# Recreate the month column after cleaning.
df["CloseMonth"] = (
    pd.to_datetime(
        df["CloseDate"],
        errors="coerce"
    )
    .dt.to_period("M")
    .astype(str)
)

print("Missing ClosePrice values:", df[target].isna().sum())
print("Prepared rows:", len(df))

Missing ClosePrice values: 0
Prepared rows: 78637


In [41]:
test_month = "2025-01"

available_training_months = sorted(
    month
    for month in df["CloseMonth"].dropna().unique()
    if month < test_month
)

train_months = available_training_months[-6:]

train = df[
    df["CloseMonth"].isin(train_months)
].copy()

test = df[
    df["CloseMonth"] == test_month
].copy()

print("Training months:", train_months)
print("Training rows:", len(train))
print("Test rows:", len(test))

Training months: ['2024-06', '2024-07', '2024-09', '2024-10', '2024-11', '2024-12']
Training rows: 70493
Test rows: 8144


In [42]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=20,
        min_samples_leaf=5,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

feature_set_information = {
    "Original features": (
        original_numeric_features,
        original_categorical_features
    ),

    "New features": (
        new_numeric_features,
        new_categorical_features
    )
}

results = []

for feature_set_name, (
    numeric_columns,
    categorical_columns
) in feature_set_information.items():

    feature_columns = (
        numeric_columns +
        categorical_columns
    )

    X_train = train[feature_columns]
    y_train = train[target]

    X_test = test[feature_columns]
    y_test = test[target]

    for model_name, model in models.items():

        preprocessor = create_preprocessor(
            numeric_columns,
            categorical_columns
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_test)

        results.append({
            "Feature Set": feature_set_name,
            "Model": model_name,
            "R²": r2_score(y_test, predictions),
            "MAE": mean_absolute_error(
                y_test,
                predictions
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_test,
                    predictions
                )
            )
        })

results_table = pd.DataFrame(results)

results_table["MAE"] = results_table["MAE"].round(0)
results_table["RMSE"] = results_table["RMSE"].round(0)
results_table["R²"] = results_table["R²"].round(4)

results_table

,Feature Set,Model,R²,MAE,RMSE
0,Original features,Linear Regression,0.6511,77493.0,1009587.0
1,Original features,Decision Tree,0.6499,76538.0,1011320.0
2,Original features,Random Forest,0.6457,68294.0,1017391.0
3,New features,Linear Regression,0.6510,77593.0,1009693.0
4,New features,Decision Tree,0.6498,76550.0,1011471.0
5,New features,Random Forest,0.6458,68943.0,1017221.0


In [43]:
def create_models():
    """
    Creates a fresh copy of each model.
    """

    return {
        "Linear Regression": LinearRegression(),

        "Decision Tree": DecisionTreeRegressor(
            max_depth=20,
            min_samples_leaf=5,
            random_state=42
        ),

        "Random Forest": RandomForestRegressor(
            n_estimators=50,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    }


feature_sets = {
    "Original features": (
        original_numeric_features,
        original_categorical_features
    ),

    "New Week 6 features": (
        new_numeric_features,
        new_categorical_features
    )
}

results = []

for feature_set_name, (
    numeric_columns,
    categorical_columns
) in feature_sets.items():

    feature_columns = (
        numeric_columns +
        categorical_columns
    )

    X_train = train[feature_columns]
    y_train = train["ClosePrice"]

    X_test = test[feature_columns]
    y_test = test["ClosePrice"]

    for model_name, model in create_models().items():

        preprocessor = create_preprocessor(
            numeric_columns,
            categorical_columns
        )

        model_pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        # Train the model.
        model_pipeline.fit(X_train, y_train)

        # Predict January 2025 prices.
        predictions = model_pipeline.predict(X_test)

        # Store model performance.
        results.append({
            "Feature Set": feature_set_name,
            "Model": model_name,
            "R²": r2_score(
                y_test,
                predictions
            ),
            "MAE": mean_absolute_error(
                y_test,
                predictions
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_test,
                    predictions
                )
            )
        })

results_table = pd.DataFrame(results)

results_table["R²"] = results_table["R²"].round(4)
results_table["MAE"] = results_table["MAE"].round(0)
results_table["RMSE"] = results_table["RMSE"].round(0)

results_table

,Feature Set,Model,R²,MAE,RMSE
0,Original features,Linear Regression,0.6511,77493.0,1009587.0
1,Original features,Decision Tree,0.6499,76538.0,1011320.0
2,Original features,Random Forest,0.6457,68294.0,1017391.0
3,New Week 6 features,Linear Regression,0.6510,77593.0,1009693.0
4,New Week 6 features,Decision Tree,0.6498,76550.0,1011471.0
5,New Week 6 features,Random Forest,0.6458,68943.0,1017221.0


In [44]:
formatted_results = results_table.copy()

formatted_results["MAE"] = formatted_results["MAE"].map(
    lambda value: f"${value:,.0f}"
)

formatted_results["RMSE"] = formatted_results["RMSE"].map(
    lambda value: f"${value:,.0f}"
)

formatted_results

,Feature Set,Model,R²,MAE,RMSE
0,Original features,Linear Regression,0.6511,"$77,493","$1,009,587"
1,Original features,Decision Tree,0.6499,"$76,538","$1,011,320"
2,Original features,Random Forest,0.6457,"$68,294","$1,017,391"
3,New Week 6 features,Linear Regression,0.6510,"$77,593","$1,009,693"
4,New Week 6 features,Decision Tree,0.6498,"$76,550","$1,011,471"
5,New Week 6 features,Random Forest,0.6458,"$68,943","$1,017,221"


Week 6 added three new types of information to the original model: a bed-to-bathroom ratio, property age, and school-district information obtained through a spatial join. The old and new feature sets were evaluated using the same training months, January 2025 test month, and three models so that the comparison would be fair. If the new feature set produces a higher R² and lower MAE or RMSE, the engineered features improved the model; otherwise, the additional features did not improve prediction performance for this dataset.

Week 7 Gradient Boost

In [45]:
from sklearn.ensemble import GradientBoostingRegressor

In [46]:
new_numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "DaysOnMarket",
    "ListPrice",
    "Latitude",
    "Longitude",
    "BedBathRatio",
    "PropertyAge"
]

new_categorical_features = [
    "City",
    "CountyOrParish",
    "SchoolDistrict"
]

new_features = (
    new_numeric_features +
    new_categorical_features
)

In [47]:
models = {
    "Gradient Boosting - Baseline": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "Gradient Boosting - Deeper": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        random_state=42
    ),

    "Gradient Boosting - More Trees": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "Gradient Boosting - Faster Learning": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.10,
        max_depth=3,
        random_state=42
    )
}

In [48]:
print("df:", "df" in globals())
print("train:", "train" in globals())
print("test:", "test" in globals())
print("new_features:", "new_features" in globals())
print("models:", "models" in globals())
print("create_preprocessor:", "create_preprocessor" in globals())

df: True
train: True
test: True
new_features: True
models: True
create_preprocessor: True


In [49]:
# Define the target and feature list.
target = "ClosePrice"

new_features = (
    new_numeric_features +
    new_categorical_features
)

# Use January 2025 as the test month.
test_month = "2025-01"

# Select all available months before January 2025.
available_training_months = sorted(
    month
    for month in df["CloseMonth"].dropna().unique()
    if month < test_month
)

# Use the six most recent months for training.
train_months = available_training_months[-6:]

# Create the training and testing datasets.
train = df[
    df["CloseMonth"].isin(train_months)
].copy()

test = df[
    df["CloseMonth"] == test_month
].copy()

print("Training months:", train_months)
print("Training rows:", len(train))
print("Test rows:", len(test))

Training months: ['2024-06', '2024-07', '2024-09', '2024-10', '2024-11', '2024-12']
Training rows: 70493
Test rows: 8144


In [50]:
results = []

X_train = train[new_features]
y_train = train["ClosePrice"]

X_test = test[new_features]
y_test = test["ClosePrice"]

for model_name, model in models.items():

    preprocessor = create_preprocessor(
        new_numeric_features,
        new_categorical_features
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    results.append({
        "Model": model_name,
        "R²": r2_score(y_test, predictions),
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(
            mean_squared_error(y_test, predictions)
        )
    })

results_table = pd.DataFrame(results)

results_table["R²"] = results_table["R²"].round(4)
results_table["MAE"] = results_table["MAE"].round(0)
results_table["RMSE"] = results_table["RMSE"].round(0)

results_table

,Model,R²,MAE,RMSE
0,Gradient Boosting - Baseline,0.6510,72156.0,1009687.0
1,Gradient Boosting - Deeper,0.6514,66059.0,1009126.0
2,Gradient Boosting - More Trees,0.6328,71631.0,1035755.0
3,Gradient Boosting - Faster Learning,0.6484,69665.0,1013454.0


In [51]:
formatted_results = results_table.copy()

formatted_results["MAE"] = formatted_results["MAE"].map(
    lambda value: f"${value:,.0f}"
)

formatted_results["RMSE"] = formatted_results["RMSE"].map(
    lambda value: f"${value:,.0f}"
)

formatted_results

,Model,R²,MAE,RMSE
0,Gradient Boosting - Baseline,0.6510,"$72,156","$1,009,687"
1,Gradient Boosting - Deeper,0.6514,"$66,059","$1,009,126"
2,Gradient Boosting - More Trees,0.6328,"$71,631","$1,035,755"
3,Gradient Boosting - Faster Learning,0.6484,"$69,665","$1,013,454"


The Gradient Boosting–Deeper model performed best, with the highest R² (0.6514) and lowest MAE ($66,059) and RMSE ($1,009,126). This suggests that deeper trees improved price predictions, while adding more trees produced worse results.